In this code, we'll be defining logistic regression class and using it to classify the drug based on given variables.

The data is taken from kaggle. Link to data: https://www.kaggle.com/datasets/prathamtripathi/drug-classification?select=drug200.csv

# Data reading and cleaning

In [273]:
# import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder, StandardScaler
import numpy as np

In [274]:
# Read the data
data = pd.read_csv('drug200.csv')
data.sample(5)

,Age,Sex,BP,Cholesterol,Na_to_K,Drug
32,49,M,LOW,NORMAL,11.014,drugX
136,55,F,HIGH,HIGH,10.977,drugB
65,68,F,NORMAL,NORMAL,27.050,DrugY
47,68,M,LOW,HIGH,10.291,drugC
175,73,F,HIGH,HIGH,18.348,DrugY


In [275]:
# Check for missing values
data.isna().sum()

# Train and Test Split
X_train, X_test, y_train, y_test = train_test_split(data.drop('Drug', axis=1), data['Drug'], test_size=0.3)

# Classify the 'Sex' column using One-Hot Encoding and 'Cholesterol' column using Ordinal Encoding
ord_enc = OrdinalEncoder(categories=[['LOW', 'NORMAL', 'HIGH'], ['NORMAL', 'HIGH']])
oh_enc = OneHotEncoder(drop='first', sparse_output=False)
oh_enc.fit(X_train[['Sex']])
X_train['Sex'] = oh_enc.transform(X_train[['Sex']])
X_test['Sex'] = oh_enc.transform(X_test[['Sex']])
X_train[['BP','Cholesterol']] = ord_enc.fit_transform(X_train[['BP','Cholesterol']])
X_test[['BP','Cholesterol']] = ord_enc.transform(X_test[['BP','Cholesterol']])

scaler = StandardScaler()
X_train[['Age','Na_to_K']] = scaler.fit_transform(X_train[['Age','Na_to_K']])
X_test[['Age','Na_to_K']] = scaler.transform(X_test[['Age','Na_to_K']])

X_train.head()

,Age,Sex,BP,Cholesterol,Na_to_K
191,-1.319157,1.0,2.0,1.0,-1.074077
135,1.777090,1.0,0.0,0.0,-0.522557
172,-0.347786,0.0,1.0,0.0,0.219636
140,0.259322,1.0,2.0,0.0,-1.318667
100,-0.833472,1.0,2.0,0.0,-0.532105


In [276]:
# Classify the y_train using following transformation: 'DrugX': 0, 'DrugY': 1, 'DrugA': 2, 'DrugB': 3, 'DrugC': 4
label_map = {'drugX': 0, 'DrugY': 1, 'drugA': 2, 'drugB': 3, 'drugC': 4}
y_test_new = y_test.map(label_map) # For accuracy calculation purpose
y_train_knn = y_train.map(label_map) # For KNN purpose 

# For gradient calculation purpose, we'll use different mapping: 'DrugX': [1, 0, 0, 0, 0], 'DrugY': [0, 1, 0, 0, 0], 'DrugA': [0, 0, 1, 0, 0], 'DrugB': [0, 0, 0, 1, 0], 'DrugC': [0, 0, 0, 0, 1]
one_hot_map = {'drugX': [1, 0, 0, 0, 0], 'DrugY': [0, 1, 0, 0, 0], 'drugA': [0, 0, 1, 0, 0], 'drugB': [0, 0, 0, 1, 0], 'drugC': [0, 0, 0, 0, 1]}
y_train_new = y_train.map(one_hot_map)
y_train_new = np.array(y_train_new.tolist())


# Logistic Regression

In [277]:
class Log_reg:
    def __init__(self):
        self.w = None

    def softmax(self, z):
        # z is a 2D array of shape (n_samples, n_classes)
        # new_z is exponent of each element in z
        new_z = np.exp(z)
        sums = np.sum(new_z, axis=1, keepdims=True)
        return new_z / sums
    
    def fit(self, X, y, lr=0.5, n_iter=1000): # After experimenting with different lr values, 0.5 was observed to give the best accuracy
        # Append 1 in the 1st column of X for bias term
        X_new = np.hstack((np.ones((X.shape[0], 1)), X))
        n_samples, n_features = X_new.shape
        self.w = np.zeros((n_features,5))
        for i in range(n_iter):
            logit = np.dot(X_new, self.w)
            y_pred = self.softmax(logit)
            dw = (1 / n_samples) * np.dot(X_new.T, (y_pred - y))
            self.w -= lr * dw
    
    def predict(self, X):
        X_new = np.hstack((np.ones((X.shape[0], 1)), X))
        y_pred = self.softmax(np.dot(X_new, self.w))
        return np.argmax(y_pred, axis=1)
    
    def accuracy(self, y_true, y_pred):
        return np.mean(y_true == y_pred)
    
# Train the model
log_regression = Log_reg()
log_regression.fit(X_train.values, y_train_new)

y_test_pred = log_regression.predict(X_test.values)
accuracy = log_regression.accuracy(y_test_new.values, y_test_pred)
print(f'Accuracy: {accuracy:.2f}')

print(f'Predicted labels: {y_test_pred}')
print(f'True labels: {y_test_new.values}')

Accuracy: 0.92
Predicted labels: [2 0 1 1 3 4 1 1 1 1 1 1 4 4 1 2 0 4 1 1 1 0 1 4 0 1 4 2 1 0 0 1 1 1 3 0 3
 1 3 1 2 1 1 0 1 0 1 1 2 1 2 1 0 1 1 1 1 0 2 1]
True labels: [2 0 1 1 3 4 1 0 1 1 1 1 4 4 1 2 4 4 1 1 1 0 1 4 4 0 4 2 1 0 0 1 1 1 3 0 3
 1 3 1 2 1 1 0 1 0 1 1 2 0 2 1 0 1 1 1 1 0 2 1]


# kNN algorithm

In [278]:
# Here we'll define class of kNN and compare the result with logistic regression model

class kNN:
    def __init__(self, X, y, k=3):
        self.k = k
        self.X = X
        self.y = y
        self.n = X.shape[0]
    
    def predict(self, X_test):
        norms = [[[np.linalg.norm(self.X[i] - x), i]  for i in range(self.n)] for x in X_test]
        # Sort each row of norms in increasing order and get the indices of the k nearest neighbors
        neighbors = [sorted(norms[i], key=lambda x: x[0])[:self.k] for i in range(len(X_test))]
        neighbour_indices = [[neighbor[1] for neighbor in neighbors[i]] for i in range(len(X_test))]
        neighbour_labels = [[self.y[neighbour_indices[i][j]] for j in range(self.k)] for i in range(len(X_test))]
        y_pred = [max(set(neighbour_labels[i]), key=neighbour_labels[i].count) for i in range(len(X_test))]
        return y_pred
    
    def accuracy(self, y_true, y_pred):
        return np.mean(y_true == y_pred)
    
knn = kNN(X_train.values, y_train_knn.values)
y_test_pred_knn = knn.predict(X_test.values)
accuracy_knn = knn.accuracy(y_test_new.values, y_test_pred_knn)
print(f'kNN Accuracy: {accuracy_knn:.2f}')
print(f'kNN Predicted labels: {y_test_pred_knn}')
print(f'kNN True labels: {y_test_new.values}')

kNN Accuracy: 0.85
kNN Predicted labels: [2, 0, 1, 1, 3, 4, 1, 0, 1, 1, 1, 1, 4, 4, 1, 2, 0, 4, 1, 1, 1, 0, 1, 4, 1, 1, 0, 2, 1, 0, 0, 1, 1, 1, 3, 0, 3, 2, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 2, 1, 2, 0, 0, 1, 1, 1, 1, 0, 2, 1]
kNN True labels: [2 0 1 1 3 4 1 0 1 1 1 1 4 4 1 2 4 4 1 1 1 0 1 4 4 0 4 2 1 0 0 1 1 1 3 0 3
 1 3 1 2 1 1 0 1 0 1 1 2 0 2 1 0 1 1 1 1 0 2 1]
